In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Load the Dataset into DataFrame

data = {
    'car_name':  ['Maruti Swift', 'Honda City', 'Ferrari 488', 'Hyundai i20',
                  'Toyota Innova', 'Maruti Baleno', 'BMW 5 Series', 'Tata Nexon',
                  'Ferrari Roma', 'Honda Amaze', 'BMW 3 Series', 'Hyundai Creta'],
    'car_age':   [5, 3, 2, 7, 4, 6, 3, 5, 1, 4, 6, 2],
    'km_driven': [45000, 28000, 5000, 72000, 38000, 55000, 22000, 48000,
                  3000, 35000, 60000, 18000],
    'engine_cc': [1197, 1498, 3902, 1197, 2694, 1197, 1998, 1497,
                  3855, 1199, 1998, 1497],
    'fuel':      ['Petrol', 'Diesel', 'Petrol', 'Petrol', 'Diesel',
                  'Petrol', 'Petrol', 'Diesel', 'Petrol', 'Petrol', 'Petrol', 'Diesel'],
    'brand':     ['Maruti', 'Honda', 'Ferrari', 'Hyundai', 'Toyota',
                  'Maruti', 'BMW', 'Tata', 'Ferrari', 'Honda', 'BMW', 'Hyundai'],
    'price':     [5.2, 8.4, 220.0, 3.8, 12.5, 4.9, 38.0, 7.1,
                  210.0, 6.8, 35.0, 11.2]
}

df = pd.DataFrame(data)

# Task 1: Data pre-processing

# a. Drop car_name

df = df.drop(columns=['car_name'])

# b. Impute missing values (Median for numerical, Mode for categorical)
num_cols = df.select_dtypes(include=[np.number]).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

cat_cols = df.select_dtypes(include=['object']).columns
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

# c. One-hot encoding
df_encoded = pd.get_dummies(df, columns=['fuel', 'brand'], drop_first=True)

# d. Separate features (X) and target (y)
X = df_encoded.drop(columns=['price'])
y = df_encoded['price']

# Task 2: Split and Scale - 80/20 split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Fit scaler on training data only
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Task 3: Train and Predict
model = LinearRegression()
model.fit(X_train_scaled, y_train)

# Predict on test set
y_pred = model.predict(X_test_scaled)

# Manual reconstruction for the first test sample
first_test_row = X_test_scaled[0]
manual_prediction = np.dot(first_test_row, model.coef_) + model.intercept_

print(f"Model Prediction (row 0): {y_pred[0]:.4f}")
print(f"Manual Reconstruction:    {manual_prediction:.4f}")

# Task 4: Evaluate and Interpret the metrics
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"\nMetrics:\nMAE: {mae:.4f} | RMSE: {rmse:.4f} | R2: {r2:.4f}")

# Interpret coefficients
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_
})
coef_df['Abs_Coefficient'] = coef_df['Coefficient'].abs()
coef_df = coef_df.sort_values(by='Abs_Coefficient', ascending=False)

print("\nFeature Importance (Coefficients):")
print(coef_df[['Feature', 'Coefficient']])

strongest_feature = coef_df.iloc[0]['Feature']
print(f"\nStrongest Influence Feature: {strongest_feature}")

Model Prediction (row 0): 32.6475
Manual Reconstruction:    32.6475

Metrics:
MAE: 1.8272 | RMSE: 1.9138 | R2: 0.9804

Feature Importance (Coefficients):
         Feature  Coefficient
4  brand_Ferrari    44.554801
2      engine_cc    36.989323
1      km_driven   -20.667246
0        car_age    19.211389
9   brand_Toyota   -13.264286
3    fuel_Petrol     1.770418
8     brand_Tata    -1.570033
7   brand_Maruti    -1.089482
5    brand_Honda    -0.710022
6  brand_Hyundai     0.545997

Strongest Influence Feature: brand_Ferrari
